# FOB Hub Netbacks vs Cargo NEA/NWE Netbacks

Comparing FOB Hub Netbacks vs Cargo Netbacks to assess the profitibality of utilising regas capacity vs selling DES

For a full explanation of how to import our FOB Hub Netbacks or Cargo Netbacks data, please refer to our Python Jupyter Notebook Code Samples:

https://www.sparkcommodities.com/api/code-examples/jupyter.html

### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

This section can remain unchanged for most Spark API users.

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from pprint import pprint
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not downloaded them already. Instructions for downloading your credentials can be found here:

https://app.sparkcommodities.com/data-integrations/api

In [ ]:
# Insert file path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

# 2. Fetching FOB Hub Netbacks

Here we define the functions used to call the FOB Hub Netbacks API data. For more information on the FOB Hub Netbacks API, please visit the API website or download our dedicated code sample:

https://www.sparkcommodities.com/api/lng-access/fob-hub-netbacks.html


In [ ]:
## Defining the reference-data calling function

def fetch_fobhub_reference_data(access_token, format='json'):

    uri = "/v1.0/lng/access/fob-hub-netbacks/reference-data/"
    print(uri)

    content = do_api_get_query(uri=uri, access_token=access_token, format=format)

    if format == 'json':
        data = content
    elif format == 'csv':
        if len(content) == 0:
            data = content
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))

    return data

In [ ]:
# fetching in CSV format (easier to browse the terminal list)
ref_data = fetch_fobhub_reference_data(access_token, format='csv')
ref_data

In [ ]:
## Defining the data calling function

def fetch_fobhub_releases(access_token, unit, fob_port_uuid, via_point, start=None, end=None, limit=None, offset=None, terminal_uuid=None, format='json'):

    query_params = "?unit={}".format(unit)

    query_params += "&fob-port-uuid={}".format(fob_port_uuid)

    if via_point is not None:
        query_params += "&via-point={}".format(via_point)

    if start is not None:
        query_params += "&start={}".format(start)
    if end is not None:
        query_params += "&end={}".format(end)
    
    if terminal_uuid is not None:
        query_params += "&regas-terminal-uuid={}".format(terminal_uuid)
    

    uri = "/v1.0/lng/access/fob-hub-netbacks/{}".format(query_params)

    content = do_api_get_query(
        uri=uri, access_token=access_token, format=format
    )

    if format == 'json':
        data = content
    elif format == 'csv':
        # if there's no data to show, returns raw response (empty string) and "No Data to Show" message
        if len(content) == 0:
            data = content
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))  # automatically converting into a Pandas DataFrame when choosing CSV format

    return data


## 3. Fetching Cargo Netbacks

Here we define the functions used to call the Cargo Netbacks API data. For more information on the Cargo Netbacks API, please visit the API website or download our dedicated code sample:

https://www.sparkcommodities.com/api/lng-cargo/netbacks.html

In [ ]:
# defining netbacks function

def fetch_netback(access_token, ticker, release=None, via=None, laden=None, ballast=None, start=None, end=None, percent_hire=None, format='json'):
    
    query_params = "?fob-port={}".format(ticker)
    if release is not None:
        query_params += "&release-date={}".format(release)
    if start is not None:
        query_params += "&start={}".format(start)
    if end is not None:
        query_params += "&end={}".format(end)
    if via is not None:
        query_params += "&via-point={}".format(via)
    if laden is not None:
        query_params += "&laden-congestion-days={}".format(laden)
    if ballast is not None:
        query_params += "&ballast-congestion-days={}".format(ballast)
    if percent_hire is not None:
        query_params += "&percent-hire={}".format(percent_hire)
    
    
    content = do_api_get_query(
        uri="/v1.0/netbacks/{}".format(query_params),
        access_token=access_token, format=format,
    )
    
    if format == 'json':
        data = content
    elif format == 'csv':
        # if there's no data to show, returns raw response (empty string) and "No Data to Show" message
        if len(content) == 0:
            data = content
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))  # automatically converting into a Pandas DataFrame when choosing CSV format

    return data


## 4. Combined extended historical data call function

A single call to the historical endpoint for both datasets is limited to 365 days of data. If you need a longer range, use the loop function below, which calls the chosen endpoint one year at a time and concatenates the results into a single Pandas DataFrame.

In [ ]:

def loop_historical_data(access_token, endpoint, fob_port_uuid=None, via_point=None, unit=None, start=None, end=None, terminal_uuid=None):
    
    hist_diff = (datetime.datetime.strptime(end, '%Y-%m-%d') - datetime.datetime.strptime(start, '%Y-%m-%d')).days
    t = 0

    starts = []
    ends = []

    w = 365

    while t < hist_diff:
        # initialising dataframe
        if t == 0 and hist_diff>w:
            diff_end = datetime.datetime.strftime(datetime.datetime.strptime(start, '%Y-%m-%d') + pd.Timedelta(days=w), '%Y-%m-%d')
            if endpoint == 'fob-hub-netbacks':
                hist_df = fetch_fobhub_releases(access_token, fob_port_uuid=fob_port_uuid, via_point=via_point,unit=unit, start=start, end=diff_end, format='csv')
            elif endpoint == 'cargo-netbacks':
                hist_df = fetch_netback(access_token, ticker=fob_port_uuid, via=via_point, start=start, end=diff_end, format='csv')

            starts.append(start)
            ends.append(diff_end)
        
        elif t==0 and hist_diff<=w:
            if endpoint == 'fob-hub-netbacks':
                hist_df = fetch_fobhub_releases(access_token, fob_port_uuid=fob_port_uuid, via_point=via_point,unit=unit, start=start, end=end, format='csv')
            elif endpoint == 'cargo-netbacks':
                hist_df = fetch_netback(access_token, ticker=fob_port_uuid, via=via_point, start=start, end=end, format='csv')

        # appending additional historical data
        else:
            if t < hist_diff-w:
                diff_start = datetime.datetime.strftime(datetime.datetime.strptime(start, '%Y-%m-%d') + pd.Timedelta(days=t+1), '%Y-%m-%d')
                diff_end = datetime.datetime.strftime(datetime.datetime.strptime(diff_start, '%Y-%m-%d') + pd.Timedelta(days=w), '%Y-%m-%d')
                if endpoint == 'fob-hub-netbacks':
                    historical_addition = fetch_fobhub_releases(access_token, fob_port_uuid=fob_port_uuid, via_point=via_point,unit=unit, start=diff_start, end=diff_end, format='csv')
                elif endpoint == 'cargo-netbacks':
                    historical_addition = fetch_netback(access_token, ticker=fob_port_uuid, via=via_point, start=diff_start, end=diff_end, format='csv')
                
                try:
                    hist_df = pd.concat([hist_df,historical_addition])
                #exception if hist_df is empty
                except:
                    hist_df = historical_addition.copy()
                starts.append(start)
                ends.append(diff_end)
            else:
                diff_start = datetime.datetime.strftime(datetime.datetime.strptime(start, '%Y-%m-%d') + pd.Timedelta(days=t+1), '%Y-%m-%d')
                diff_end = datetime.datetime.strftime(datetime.datetime.strptime(diff_start, '%Y-%m-%d') + pd.Timedelta(days=hist_diff-t), '%Y-%m-%d')

                if endpoint == 'fob-hub-netbacks':
                    historical_addition = fetch_fobhub_releases(access_token, fob_port_uuid=fob_port_uuid, via_point=via_point,unit=unit, start=diff_start, end=diff_end, format='csv')
                elif endpoint == 'cargo-netbacks':
                    historical_addition = fetch_netback(access_token, ticker=fob_port_uuid, via=via_point, start=diff_start, end=diff_end, format='csv')

                hist_df = pd.concat([hist_df, historical_addition])
                starts.append(start)
                ends.append(diff_end)
        
        #looping by year
        t += w

    if endpoint=='fob-hub-netbacks':
        for c in list(hist_df.columns)[13:]:
            hist_df[c] = pd.to_numeric(hist_df[c])
        hist_df['LoadDate'] = pd.to_datetime(hist_df['LoadDate'])
    elif endpoint == 'cargo-netbacks':
        for c in list(hist_df.columns)[4:]:
            hist_df[c] = pd.to_numeric(hist_df[c])
    hist_df['ReleaseDate'] = pd.to_datetime(hist_df['ReleaseDate'])

    return hist_df

# 4.1 Time-range Input

In [ ]:
# define today as the end date
today = datetime.datetime.now().strftime('%Y-%m-%d')
hist_end=today

# Define start date by time interval or manually
#hist_start = (datetime.datetime.now() - datetime.timedelta(days=365)).strftime('%Y-%m-%d')
hist_start = '2026-04-01'

# 4.2 Using the function to call FOB Hub Netbacks

In [ ]:
# Fetching a FoB Port UUID
sabine_uuid = ref_data[ref_data['FOBPortName'] == 'Sabine Pass']['FOBPortUUID'].iloc[0]

# Fetching data using UUID
fobhub_df = loop_historical_data(access_token, endpoint='fob-hub-netbacks', unit='usd-per-mmbtu', fob_port_uuid=sabine_uuid, via_point=None, 
                                            start=hist_start, end=hist_end)

print('Fetched {} rows from {} to {}'.format(len(fobhub_df), hist_start, hist_end))
fobhub_df.head(5)

## 4.3 Sinking Regas Costs Components for FOB Hub Netbacks

Output columns added to `hist_df`:
- `NetbackTTFBasisSunk`
- `NetbackOutrightSunk`


In [ ]:
# Global sunk slot components - added back for every terminal
sunk_slot_cols = ['SlotLtCapacityEstimate', 'SlotUnloadStorageRegas', 'SlotBerth', 'SlotBerthUnloadStorageRegas']

# Per-terminal extra sunk costs - layered on top of the global sunk costs defined above
# Value is a list of cost columns to treat as sunk for that terminal only
# Edit this dict to suit your analysis. Unlisted terminals use the global set.

sunk_terminal_costs = {
    # 'Isle of Grain': ['EntryCapacity', 'EntryVariable'],
    # 'South Hook': ['EntryCapacity'],
}

# Sink global sunk costs across all terminals
sunk_add_back = fobhub_df[sunk_slot_cols].fillna(0).sum(axis=1)

# Sink terminal-specific sunk costs
for terminal, cols in sunk_terminal_costs.items():
    if not cols:
        continue
    missing = [c for c in cols if c not in fobhub_df.columns]
    if missing:
        raise KeyError('sunk_terminal_costs[{!r}] references missing columns: {}'.format(terminal, missing))
    mask = fobhub_df['RegasTerminalName'] == terminal
    if mask.any():
        sunk_add_back.loc[mask] += fobhub_df.loc[mask, cols].fillna(0).sum(axis=1)

fobhub_df['NetbackTTFBasisSunk'] = fobhub_df['NetbackTTFBasis'] + sunk_add_back
fobhub_df['NetbackOutrightSunk'] = fobhub_df['NetbackOutright'] + sunk_add_back

fobhub_df[['ReleaseDate', 'RegasTerminalName', 'LoadMonthIndex', 'NetbackTTFBasis',
         'NetbackTTFBasisSunk']].head(10)


### Pivot into a wide DataFrame - one column per terminal

Here we pivot the dataframe so that each terminal is one column. We create two dataframes - one with all Regas costs included, and one with defined Sunk costs excluded.

In [ ]:
# one row per (ReleaseDate, DeliveryMonth), one column per terminal
terminal_list = list(fobhub_df['RegasTerminalName'].unique())

id_cols = ['ReleaseDate', 'LoadDate', 'LoadMonthName', 'LoadMonthIndex']

# All regas costs included
fobhub_full = (
    fobhub_df
    .pivot_table(index=id_cols, columns='RegasTerminalName', values='NetbackTTFBasis', aggfunc='first')
    .reset_index()
)

# Defined sunk costs excluded
fobhub_sunk = (
    fobhub_df
    .pivot_table(index=id_cols, columns='RegasTerminalName', values='NetbackTTFBasisSunk', aggfunc='first')
    .reset_index()
)

# Reset column names
fobhub_full.columns.name = None
fobhub_sunk.columns.name = None

# restore terminal-appearance ordering (pivot_table sorts alphabetically by default)
fobhub_full = fobhub_full[id_cols + terminal_list]
fobhub_sunk = fobhub_sunk[id_cols + terminal_list]

fobhub_full.head(5)

## 4.4 Calling Cargo Netbacks

In [ ]:
# Calling US netbacks via COGH and via Panama
sabcogh_df = loop_historical_data(access_token, endpoint='cargo-netbacks', fob_port_uuid=sabine_uuid, via_point='cogh', 
                                            start=hist_start, end=hist_end)

sabpanama_df = loop_historical_data(access_token, endpoint='cargo-netbacks', fob_port_uuid=sabine_uuid, via_point='panama', 
                                            start=hist_start, end=hist_end)

# Combining into one dataframe
cargo_df = pd.merge(
    sabcogh_df[["ReleaseDate", "LoadMonthIndex", 'NeaNetbackOutright', 'NeaNetbackTtfBasis','DeltaNeaNwe']],
    sabpanama_df[["ReleaseDate", "LoadMonthIndex", 'NeaNetbackOutright', 'NeaNetbackTtfBasis', 'DeltaNeaNwe']],
    how="outer",
    left_on=["ReleaseDate", "LoadMonthIndex"],
    right_on=["ReleaseDate", "LoadMonthIndex"],
    suffixes=("_SabCOGH", "_SabPanama"),
)

cargo_df['LoadMonthIndexStr'] = 'M+' + cargo_df['LoadMonthIndex'].astype(str)
cargo_df.head(3)

## 4.5 Creating combined Dataframe

In [ ]:
# All regas costs included
combined_terminals_full = pd.merge(
    fobhub_full,
    cargo_df[["ReleaseDate", "LoadMonthIndexStr", 'NeaNetbackTtfBasis_SabCOGH', 'NeaNetbackTtfBasis_SabPanama']],
    how="left",
    left_on=["ReleaseDate", "LoadMonthIndex"],
    right_on=["ReleaseDate", "LoadMonthIndexStr"],
)

combined_terminals_full['LoadDate'] = pd.to_datetime(combined_terminals_full['LoadDate'])

# Defined sunk costs excluded
combined_terminals_sunk = pd.merge(
    fobhub_sunk,
    cargo_df[["ReleaseDate", "LoadMonthIndexStr", 'NeaNetbackTtfBasis_SabCOGH', 'NeaNetbackTtfBasis_SabPanama']],
    how="left",
    left_on=["ReleaseDate", "LoadMonthIndex"],
    right_on=["ReleaseDate", "LoadMonthIndexStr"],
)

combined_terminals_sunk['LoadDate'] = pd.to_datetime(combined_terminals_sunk['LoadDate'])

## 5. Creating equivalent Country-Averaged datasets

Country averaging FOB Hub Netbacks data as an alternative, "zoomed-out" dataset. 

__N.B.__ Section 5 can be deleted if you intend to only use terminal specific data.

In [ ]:
# Country mapping dictionary
terminal_country_dict = {
    'Netherlands': ['Gate', 'EemsEnergyTerminal'],
    'UK':          ['Isle of Grain', 'South Hook'],
    'Belgium':     ['Zeebrugge'],
    'France':      ['Dunkerque', 'Le Havre', 'Montoir', 'Fos Cavaou'],
    'Germany':     ['Brunsbuttel', 'Deutsche Ostsee', 'Deutsche Ostsee Phase 1',
                    'Wilhelmshaven 1', 'Wilhelmshaven 2', 'Stade'],
    'Italy':       ['Adriatic', 'OLT Toscana', 'Piombino', 'Ravenna'],
    'Spain':       ['Spain TVB'],
}

### Country-level Averages

`countries_full` and `countries_sunk` average the per-terminal columns in `fobhub_full` / `fobhub_sunk` up to country level using `terminal_country_dict`.

In [ ]:
# Columns to initialise DataFrame with
id_cols = ['ReleaseDate', 'LoadDate', 'LoadMonthName', 'LoadMonthIndex']

# Iterating over terminals to see which have a country assigned to them, and filtering accordingly
country_to_present_terminals = {}
partial_warnings = []

for country, terms in terminal_country_dict.items():
    present = [t for t in terms if t in fobhub_full.columns]
    missing = [t for t in terms if t not in fobhub_full.columns]
    if not present:
        partial_warnings.append('  - {}: SKIPPED (no mapped terminals in fobhub_full). Missing: {}'.format(country, missing))
        continue
    if missing:
        partial_warnings.append('  - {}: averaging {} (missing from data: {})'.format(country, present, missing))
    country_to_present_terminals[country] = present

if partial_warnings:
    print('Country mapping notes:')
    for w in partial_warnings:
        print(w)

# Aggregating by country for all terminals
def _aggregate_by_country(df, country_map, id_cols):
    out = df[id_cols].copy()
    for country, terms in country_map.items():
        out[country] = df[terms].mean(axis=1, skipna=True)
    return out

countries_full = _aggregate_by_country(fobhub_full, country_to_present_terminals, id_cols)
countries_sunk = _aggregate_by_country(fobhub_sunk, country_to_present_terminals, id_cols)

# round to 3 d.p. for readability
country_cols = list(country_to_present_terminals.keys())
countries_full[country_cols] = countries_full[country_cols].round(3)
countries_sunk[country_cols] = countries_sunk[country_cols].round(3)

countries_full.head()

### Combining the Countries dataframes with the Cargo Netbacks to make one aligned dataframe for ease of comparison & plotting

In [ ]:
countries_full = pd.merge(
    countries_full,
    cargo_df[["ReleaseDate", "LoadMonthIndexStr", 'NeaNetbackTtfBasis_SabCOGH', 'NeaNetbackTtfBasis_SabPanama']],
    how="left",
    left_on=["ReleaseDate", "LoadMonthIndex"],
    right_on=["ReleaseDate", "LoadMonthIndexStr"],
)

countries_sunk = pd.merge(
    countries_sunk,
    cargo_df[["ReleaseDate", "LoadMonthIndexStr", 'NeaNetbackTtfBasis_SabCOGH', 'NeaNetbackTtfBasis_SabPanama']],
    how="left",
    left_on=["ReleaseDate", "LoadMonthIndex"],
    right_on=["ReleaseDate", "LoadMonthIndexStr"],
)

countries_sunk.head(5)

# 6. Plotting 

## 6.1. Front Month Moneyness Evolution & Latest Forward Curve for several terminals

In [ ]:
# Creating front month and forward curve dataframes, for both "full regas" and "sunk costs excl." datasets
front_full = combined_terminals_full[combined_terminals_full['LoadMonthIndex'] == 'M+1'].copy()
front_sunk = combined_terminals_sunk[combined_terminals_sunk['LoadMonthIndex'] == 'M+1'].copy()

latest = combined_terminals_full['ReleaseDate'].unique()[-1]
latest_full = combined_terminals_full[combined_terminals_full['ReleaseDate'] == latest].copy()
latest_sunk = combined_terminals_sunk[combined_terminals_sunk['ReleaseDate'] == latest].copy()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

fig, ax= plt.subplots(figsize=(14,7))
fig.subplots_adjust(top=0.78, bottom=0.10, left=0.06, right=0.96)

# User input for which countries to plot, and associated plot colours
my_terminals = ['Deutsche Ostsee', 'Brunsbuttel', 'Gate', 'Isle of Grain', 'South Hook', 'Fos Cavaou', 'Montoir', 'OLT Toscana', 'Spain TVB']
colours = ['orange', 'sienna', 'grey', '#0086ad', '#011f4b', '#00c2c7', 'salmon', 'mediumseagreen', 'firebrick']

print("Latest Assessment: " + str(front_full['ReleaseDate'].iloc[0]))

# Iterating over all terminals & plotting
cz = 0
for t in my_terminals:

    print(t + ' = ' + str(front_sunk[t].iloc[-1]))
    
    ax.scatter(front_sunk['ReleaseDate'].iloc[-1],  front_sunk[t].iloc[-1], 
                color=colours[cz], marker='o', s=80)
    ax.plot(front_sunk['ReleaseDate'], front_sunk[t], 
            linewidth=2.5, label=my_terminals[cz], color=colours[cz])
    ax.plot(latest_sunk['LoadDate'].iloc[1:], latest_sunk[t].iloc[1:], 
            linewidth=2.0, marker='s', linestyle='--', color=colours[cz])

    if cz == 0:
        # Plot US via Panama NE-Asia Netback
        ax.scatter(front_sunk['ReleaseDate'].iloc[-1],  front_sunk['NeaNetbackTtfBasis_SabPanama'].iloc[-1], 
                    color='#4F41F4', marker='o', s=80)
        ax.plot(front_sunk['ReleaseDate'], front_sunk['NeaNetbackTtfBasis_SabPanama'], 
                linewidth=2.5, label='NEA via Panama', color='#4F41F4')
        ax.plot(latest_sunk['LoadDate'].iloc[1:], latest_sunk['NeaNetbackTtfBasis_SabPanama'].iloc[1:], 
                linewidth=2.0, marker='s', linestyle='-', color='#4F41F4')
        
        # Plot US via COGH NE-Asia Netback
        #ax.scatter(front_sunk['ReleaseDate'].iloc[-1],  front_sunk['NeaNetbackTtfBasis_SabPanama'].iloc[-1], 
        #           color='#4F41F4', marker='o', s=80)
        #ax.plot(front_sunk['ReleaseDate'], front_sunk['NeaNetbackTtfBasis_SabPanama'], 
        #       linewidth=2.5, label='NEA via COGH', color='#4F41F4')
        #ax.plot(latest_full['LoadDate'].iloc[1:], latest_sunk['NeaNetbackTtfBasis_SabPanama'].iloc[1:], 
        #       linewidth=2.0, marker='s', linestyle='-', color='#4F41F4')
    
    cz+=1

plt.tight_layout()

ax.legend(loc='lower right', frameon=True, edgecolor='#dddddd', fontsize=10.5)

plt.show()

## 6.2.  European Regas vs Asia Arb

Taking the difference of each terminals FOB Hub Netback and the Cargo Netback to Asia (in this case, via Panama) to calculate a "Regas Arb" vs Asia

In [ ]:
fig, ax= plt.subplots(figsize=(14,7))
fig.subplots_adjust(top=0.78, bottom=0.10, left=0.06, right=0.96)

# User input for which countries to plot, and associated plot colours
my_terminals = ['Deutsche Ostsee', 'Brunsbuttel', 'Gate', 'Isle of Grain', 'South Hook', 'Fos Cavaou', 'Montoir', 'OLT Toscana', 'Spain TVB']
colours = ['orange', 'sienna', 'grey', '#0086ad', '#011f4b', '#00c2c7', 'salmon', 'mediumseagreen', 'firebrick']

print("Latest Assessment: " + str(front_full['ReleaseDate'].iloc[0]))

# Iterating over all terminals to plot
cz = 0
for t in my_terminals:

    print(t + ' = ' + str(front_sunk[t].iloc[-1]))
    
    ax.scatter(front_sunk['ReleaseDate'].iloc[-1],  front_sunk[t].iloc[-1] - front_sunk['NeaNetbackTtfBasis_SabPanama'].iloc[-1], 
                color=colours[cz], marker='o', s=80)
    ax.plot(front_sunk['ReleaseDate'], front_sunk[t] - front_sunk['NeaNetbackTtfBasis_SabPanama'], 
            linewidth=2.5, label=my_terminals[cz], color=colours[cz])
    ax.plot(latest_sunk['LoadDate'].iloc[1:], latest_sunk[t].iloc[1:] - latest_sunk['NeaNetbackTtfBasis_SabPanama'].iloc[1:], 
            linewidth=2.0, marker='s', linestyle='--', color=colours[cz])
    
    cz+=1

# Plot y=0 line to make the In the Money/Out the Money boundary clear
ax.hlines(0, front_sunk['ReleaseDate'].iloc[0], latest_sunk['LoadDate'].iloc[-1]+pd.Timedelta(days=20), color='grey', zorder=0)

# Conditional <0 red shading
negrange = [front_sunk['ReleaseDate'].iloc[0] -pd.Timedelta(days=5), latest_sunk['LoadDate'].iloc[-1]+pd.Timedelta(days=5)]
ax.fill_between(negrange, 0, -30, color='red', alpha=0.05)

# Axis params
plt.xlim([front_sunk['ReleaseDate'].iloc[0]-pd.Timedelta(days=1), latest_sunk['LoadDate'].iloc[-1]+pd.Timedelta(days=5)])
plt.ylim(-4, 2)
plt.tight_layout()
ax.legend(loc='lower right', frameon=True, edgecolor='#dddddd', fontsize=9.5)

plt.show()

### Below are several additional options:

- Use the "countries_full" and "countries_sunk" dataframes to plot for country averaged FOB Hub Netbacks (although "front_" and "latest_" dataframes will need to be defined above for the country-averaged data)
- use the "front_full" and "latest_full" dataframes to plot the FOB Hub Netbacks with all regas costs included